In [ ]:
"""
Campaign causal analysis pipeline.

Implements, in order:
  Part 1 : estimand definition (see project-breakdown-part1.md) + episode table
  Step 3 : transaction unit / zero audit
  Outcome construction (Y columns) from the transaction file
  Plan 1 : matched stacked event-study Difference-in-Differences

Assumptions baked in from prior analysis of the files (see project-breakdown-part1.md):
  - Treatment D=1 is CAMPAIGN ASSIGNMENT (campaign_table.csv), never redemption.
  - coupon.csv must be joined on (CAMPAIGN, COUPON_UPC), and deduplicated on
    (CAMPAIGN, COUPON_UPC, PRODUCT_ID) before counting eligible products.
  - Households linked to overlapping campaigns are flagged, not silently pooled.
  - Effects are reported per campaign-week, not raw campaign totals, because
    campaign duration ranges 33-162 days.

This has NOT been run against real data yet -- it was written against the
documented schemas only. Run scripts/smoke_test in this file's __main__
block after pointing CONFIG at your actual files, and inspect intermediate
outputs before trusting anything downstream.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("campaign_pipeline")


# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #

@dataclass
class Config:
    data_dir: Path
    campaign_desc_file: str = "campaign_desc.csv"
    campaign_table_file: str = "campaign_table.csv"
    coupon_file: str = "coupon.csv"
    coupon_redempt_file: str = "coupon_redempt.csv"
    product_file: str = "product.csv"
    hh_demographic_file: str = "hh_demographic.csv"
    transaction_file: str = "transaction_data.csv"

    # Analysis parameters -- these are choices, not facts from the files.
    # State them explicitly so they're easy to challenge/change.
    pre_period_weeks: int = 4          # weeks before campaign start used as baseline
    post_period_weeks: int = 4         # weeks after campaign end used for payback check
    transaction_chunksize: int = 1_000_000  # for the ~6M row file

    paths: dict = field(init=False)

    def __post_init__(self):
        self.paths = {
            "campaign_desc": self.data_dir / self.campaign_desc_file,
            "campaign_table": self.data_dir / self.campaign_table_file,
            "coupon": self.data_dir / self.coupon_file,
            "coupon_redempt": self.data_dir / self.coupon_redempt_file,
            "product": self.data_dir / self.product_file,
            "hh_demographic": self.data_dir / self.hh_demographic_file,
            "transaction": self.data_dir / self.transaction_file,
        }


# --------------------------------------------------------------------------- #
# 1. Loading
# --------------------------------------------------------------------------- #

def load_reference_tables(cfg: Config) -> dict[str, pd.DataFrame]:
    """Load the six small reference CSVs (not the transaction file)."""
    log.info("Loading reference tables")

    campaign_desc = pd.read_csv(cfg.paths["campaign_desc"])
    campaign_table = pd.read_csv(cfg.paths["campaign_table"])
    coupon = pd.read_csv(cfg.paths["coupon"])
    coupon_redempt = pd.read_csv(cfg.paths["coupon_redempt"])
    product = pd.read_csv(cfg.paths["product"])
    hh_demographic = pd.read_csv(cfg.paths["hh_demographic"])

    # Normalize column names defensively (source files have been observed to
    # vary in case/whitespace across this dataset family).
    for df in (campaign_desc, campaign_table, coupon, coupon_redempt, product, hh_demographic):
        df.columns = [c.strip().upper() for c in df.columns]

    return {
        "campaign_desc": campaign_desc,
        "campaign_table": campaign_table,
        "coupon": coupon,
        "coupon_redempt": coupon_redempt,
        "product": product,
        "hh_demographic": hh_demographic,
    }


def dedupe_coupon_table(coupon: pd.DataFrame) -> pd.DataFrame:
    """
    coupon.csv is an eligibility table with 5,164 exact-duplicate rows (4.15%)
    across (CAMPAIGN, COUPON_UPC, PRODUCT_ID) triplets. Drop exact duplicates
    only -- do NOT collapse legitimate one-coupon-to-many-product mappings.
    """
    before = len(coupon)
    coupon_dedup = coupon.drop_duplicates(
        subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]
    ).copy()
    dropped = before - len(coupon_dedup)
    log.info(f"coupon.csv: dropped {dropped} exact-duplicate rows ({dropped/before:.2%})")
    return coupon_dedup


# --------------------------------------------------------------------------- #
# 2. Episode table (Part 1 deliverable)
# --------------------------------------------------------------------------- #

def build_episode_table(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Build the household x campaign episode table.
    D (treatment indicator) = row exists here = household assigned to campaign.
    Redemption is kept as a separate outcome-adjacent column, never as D.
    """
    log.info("Building episode table")

    campaign_table = tables["campaign_table"]
    campaign_desc = tables["campaign_desc"]
    coupon = dedupe_coupon_table(tables["coupon"])
    coupon_redempt = tables["coupon_redempt"]
    hh_demographic = tables["hh_demographic"]

    episodes = campaign_table.merge(
        campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_DAY", "END_DAY"]].rename(
            columns={"DESCRIPTION": "CAMPAIGN_TYPE"}
        ),
        on="CAMPAIGN",
        how="left",
        validate="many_to_one",
    )
    episodes["DURATION_DAYS"] = episodes["END_DAY"] - episodes["START_DAY"] + 1

    # --- concurrent campaigns: count of *other* campaigns this household is
    # also linked to, whose windows overlap this campaign's window.
    episodes = _add_concurrent_campaign_count(episodes, campaign_desc)

    # --- eligible product count per campaign (post-dedup)
    eligible_counts = (
        coupon.groupby("CAMPAIGN")["PRODUCT_ID"].nunique().rename("ELIGIBLE_PRODUCT_COUNT")
    )
    episodes = episodes.merge(eligible_counts, on="CAMPAIGN", how="left")

    # --- redemption flag + first redemption day (household x campaign)
    redempt_agg = (
        coupon_redempt.groupby(["household_key".upper(), "CAMPAIGN"])["DAY"]
        .min()
        .rename("FIRST_REDEMPTION_DAY")
        .reset_index()
    )
    episodes = episodes.merge(redempt_agg, on=["HOUSEHOLD_KEY", "CAMPAIGN"], how="left")
    episodes["REDEEMED"] = episodes["FIRST_REDEMPTION_DAY"].notna().astype(int)

    # --- demographics (kept as explicit categories; missing stays missing)
    episodes = episodes.merge(hh_demographic, on="HOUSEHOLD_KEY", how="left")
    episodes["HAS_DEMOGRAPHICS"] = episodes["AGE_DESC"].notna().astype(int) if "AGE_DESC" in episodes else np.nan

    log.info(f"Episode table: {len(episodes)} rows, {episodes['HOUSEHOLD_KEY'].nunique()} households")
    return episodes


def _add_concurrent_campaign_count(episodes: pd.DataFrame, campaign_desc: pd.DataFrame) -> pd.DataFrame:
    """
    For every episode (household, campaign), count how many OTHER campaigns
    that same household is linked to whose [START_DAY, END_DAY] overlaps this
    campaign's window. O(n_households * campaigns_per_household^2) via groupby
    -- fine at 7,208 rows / max 17 campaigns per household.
    """
    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]

    def _count_for_group(group: pd.DataFrame) -> pd.Series:
        camps = group["CAMPAIGN"].tolist()
        counts = []
        for c in camps:
            s0, e0 = windows.loc[c, "START_DAY"], windows.loc[c, "END_DAY"]
            n_overlap = 0
            for other in camps:
                if other == c:
                    continue
                s1, e1 = windows.loc[other, "START_DAY"], windows.loc[other, "END_DAY"]
                if s0 <= e1 and s1 <= e0:
                    n_overlap += 1
            counts.append(n_overlap)
        return pd.Series(counts, index=group.index)

    episodes = episodes.copy()
    episodes["N_CONCURRENT_CAMPAIGNS"] = (
        episodes.groupby("HOUSEHOLD_KEY", group_keys=False).apply(_count_for_group)
    )
    return episodes


# --------------------------------------------------------------------------- #
# 3. Transaction audit (Step 3)
# --------------------------------------------------------------------------- #

def audit_transactions(cfg: Config) -> dict:
    """
    Streams the transaction file in chunks (it's ~6M rows) and reports:
      - rows/units carried by extreme-quantity outliers (possible fuel mixing)
      - household-week combinations with zero shopping trips
      - basic negative/zero-value integrity checks
    Does not modify the file; returns a summary dict to inform filtering
    decisions made explicitly later (never silently).
    """
    log.info("Auditing transaction file (chunked)")

    qty_values = []
    total_rows = 0
    total_qty = 0.0
    neg_sales = 0
    neg_qty = 0
    hh_week_pairs = set()
    all_households = set()
    all_weeks = set()

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        total_rows += len(chunk)
        total_qty += chunk["QUANTITY"].sum()
        neg_sales += (chunk["SALES_VALUE"] < 0).sum()
        neg_qty += (chunk["QUANTITY"] < 0).sum()

        qty_values.append(chunk["QUANTITY"].to_numpy())

        pairs = set(zip(chunk["household_key"], chunk["WEEK_NO"]))
        hh_week_pairs |= pairs
        all_households |= set(chunk["household_key"].unique())
        all_weeks |= set(chunk["WEEK_NO"].unique())

    qty_all = np.concatenate(qty_values)
    q99 = np.quantile(qty_all, 0.99)
    outlier_mask = qty_all >= q99
    outlier_row_share = outlier_mask.mean()
    outlier_unit_share = qty_all[outlier_mask].sum() / qty_all.sum()

    n_possible_hh_weeks = len(all_households) * len(all_weeks)
    no_trip_share = 1 - (len(hh_week_pairs) / n_possible_hh_weeks) if n_possible_hh_weeks else np.nan

    summary = {
        "total_rows": total_rows,
        "total_quantity": float(total_qty),
        "negative_sales_rows": int(neg_sales),
        "negative_quantity_rows": int(neg_qty),
        "top_1pct_qty_row_share": float(outlier_row_share),
        "top_1pct_qty_unit_share": float(outlier_unit_share),
        "n_households": len(all_households),
        "n_weeks": len(all_weeks),
        "household_week_no_trip_share": float(no_trip_share),
    }
    log.info(f"Audit summary: {summary}")
    log.warning(
        "top_1pct_qty_unit_share above should be compared against the PDF's claimed "
        "98.7%-units-in-1.2%-of-rows fuel-mixing figure. If far lower, this dataset's "
        "fuel contamination may differ from the PDF's reference dataset -- do not assume "
        "it transfers."
    )
    return summary


# --------------------------------------------------------------------------- #
# 4. Outcome construction (Y columns for the episode table)
# --------------------------------------------------------------------------- #

def compute_household_campaign_outcomes(
    cfg: Config,
    campaign_desc: pd.DataFrame,
    coupon: pd.DataFrame,
    product: pd.DataFrame,
    households_in_scope: set,
) -> pd.DataFrame:
    """
    Streams the transaction file ONCE and computes, for EVERY household in
    households_in_scope x EVERY campaign (not just linked pairs), outcomes
    over that campaign's calendar window:
      Y_ELIGIBLE_SALES / Y_ELIGIBLE_UNITS : sales of THAT campaign's eligible products
      Y_CATEGORY_SALES                    : sales in the SAME COMMODITY as that
                                             campaign's eligible products (not all sales)
      Y_RIVAL_SALES                       : Y_CATEGORY_SALES minus Y_ELIGIBLE_SALES
                                             (same commodity, non-eligible products)
      Y_PRE_SALES / Y_POST_SALES          : household total sales in the pre/post windows

    Computing this for every household (linked or not) against every campaign
    is what makes Plan 1's control group comparable: a control household's
    "during" outcome is now measured over the SAME calendar window as the
    campaign it's being matched against, not over its own unrelated episode.

    30 campaigns x ~2,500 households is a small enough combination to hold in
    memory; only the streaming transaction read needs chunking.
    """
    log.info("Computing household x campaign window outcomes (universal table)")

    # Eligible products per campaign, and the commodities those products belong to.
    eligible = coupon[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]
    eligible = eligible.assign(COMMODITY_DESC=eligible["PRODUCT_ID"].map(product_commodity))

    eligible_products_by_campaign = eligible.groupby("CAMPAIGN")["PRODUCT_ID"].apply(set).to_dict()
    eligible_commodities_by_campaign = (
        eligible.groupby("CAMPAIGN")["COMMODITY_DESC"].apply(lambda s: set(s.dropna())).to_dict()
    )

    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]
    campaigns = windows.index.tolist()

    accum = {col: {} for col in [
        "Y_ELIGIBLE_SALES", "Y_ELIGIBLE_UNITS", "Y_CATEGORY_SALES",
        "Y_RIVAL_SALES", "Y_PRE_SALES", "Y_POST_SALES",
    ]}

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk[chunk["household_key"].isin(households_in_scope)]
        if chunk.empty:
            continue
        chunk["COMMODITY_DESC"] = chunk["PRODUCT_ID"].map(product_commodity)

        # Vectorized per-campaign (30 iterations), not per-household (thousands
        # of iterations) -- each iteration is a handful of pandas groupby-sums
        # over the whole chunk rather than one over a tiny per-household slice.
        for campaign in campaigns:
            start, end = windows.loc[campaign, "START_DAY"], windows.loc[campaign, "END_DAY"]
            pre_start = start - cfg.pre_period_weeks * 7
            post_end = end + cfg.post_period_weeks * 7

            elig_products = eligible_products_by_campaign.get(campaign, set())
            elig_commodities = eligible_commodities_by_campaign.get(campaign, set())

            during = chunk[(chunk["DAY"] >= start) & (chunk["DAY"] <= end)]
            pre = chunk[(chunk["DAY"] >= pre_start) & (chunk["DAY"] < start)]
            post = chunk[(chunk["DAY"] > end) & (chunk["DAY"] <= post_end)]

            if len(during):
                is_elig = during["PRODUCT_ID"].isin(elig_products)
                same_commodity = during["COMMODITY_DESC"].isin(elig_commodities)

                elig_sales_by_hh = during.loc[is_elig].groupby("household_key")["SALES_VALUE"].sum()
                elig_units_by_hh = during.loc[is_elig].groupby("household_key")["QUANTITY"].sum()
                category_sales_by_hh = during.loc[same_commodity].groupby("household_key")["SALES_VALUE"].sum()

                for hh, val in elig_sales_by_hh.items():
                    accum["Y_ELIGIBLE_SALES"][(hh, campaign)] = accum["Y_ELIGIBLE_SALES"].get((hh, campaign), 0) + val
                for hh, val in elig_units_by_hh.items():
                    accum["Y_ELIGIBLE_UNITS"][(hh, campaign)] = accum["Y_ELIGIBLE_UNITS"].get((hh, campaign), 0) + val
                for hh, val in category_sales_by_hh.items():
                    key = (hh, campaign)
                    accum["Y_CATEGORY_SALES"][key] = accum["Y_CATEGORY_SALES"].get(key, 0) + val
                    accum["Y_RIVAL_SALES"][key] = accum["Y_RIVAL_SALES"].get(key, 0) + val - elig_sales_by_hh.get(hh, 0)

            if len(pre):
                pre_sales_by_hh = pre.groupby("household_key")["SALES_VALUE"].sum()
                for hh, val in pre_sales_by_hh.items():
                    key = (hh, campaign)
                    accum["Y_PRE_SALES"][key] = accum["Y_PRE_SALES"].get(key, 0) + val
            if len(post):
                post_sales_by_hh = post.groupby("household_key")["SALES_VALUE"].sum()
                for hh, val in post_sales_by_hh.items():
                    key = (hh, campaign)
                    accum["Y_POST_SALES"][key] = accum["Y_POST_SALES"].get(key, 0) + val

    all_keys = set()
    for d in accum.values():
        all_keys |= set(d.keys())
    idx = pd.MultiIndex.from_tuples(sorted(all_keys), names=["HOUSEHOLD_KEY", "CAMPAIGN"])
    out = pd.DataFrame(index=idx)
    for col, d in accum.items():
        out[col] = pd.Series(d)
    out = out.fillna(0.0).reset_index()

    log.info(f"Universal outcome table: {len(out)} household x campaign rows for {len(households_in_scope)} households")
    return out


# --------------------------------------------------------------------------- #
# 5. Plan 1: matched stacked event-study DiD
# --------------------------------------------------------------------------- #

def run_plan1_event_study_did(
    episodes: pd.DataFrame,
    universal_outcomes: pd.DataFrame,
    demographics: pd.DataFrame,
    outcome_col: str = "Y_ELIGIBLE_SALES",
) -> pd.DataFrame:
    """
    For each campaign:
      1. Treated = households assigned to this campaign with N_CONCURRENT_CAMPAIGNS == 0
         (concurrently-treated households are excluded from Plan 1, not modeled --
         Plan 1 can't separate simultaneous treatments).
      2. Controls = every OTHER household in universal_outcomes that is NOT linked
         to this campaign in episodes, regardless of what campaigns they're linked
         to elsewhere -- their outcome is pulled from universal_outcomes for THIS
         campaign's calendar window, so treated and control are compared over the
         identical time period. (Controls linked to a campaign that overlaps this
         one's window are still excluded, since their own treatment would confound
         the comparison.)
      3. Match treated to controls on available demographics + pre-period sales
         (nearest-neighbor on a propensity score from logistic regression).
      4. DiD estimate = (during_treated - pre_treated) - (during_control - pre_control),
         normalized to per-week.
    Returns one row per campaign with the DiD estimate and a naive matched-pair SE
    (a placeholder -- the PDF wants bootstrap/Fieller-style intervals at the ROI
    stage, not this SE; this is enough to rank campaigns directionally, not to
    report as a final interval).
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import NearestNeighbors

    log.info("Running Plan 1: matched stacked event-study DiD")
    results = []

    # household -> set of campaigns they're linked to, for exclusion checks
    linked_campaigns_by_hh = episodes.groupby("HOUSEHOLD_KEY")["CAMPAIGN"].apply(set).to_dict()
    concurrent_free_hh = set(episodes.loc[episodes["N_CONCURRENT_CAMPAIGNS"].eq(0), "HOUSEHOLD_KEY"])

    windows = episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")[["START_DAY", "END_DAY", "DURATION_DAYS"]]
    all_campaign_windows = windows[["START_DAY", "END_DAY"]]

    demo_cols = [c for c in demographics.columns if c.endswith("_DESC")]
    uo = universal_outcomes.merge(demographics, on="HOUSEHOLD_KEY", how="left")

    for campaign in windows.index:
        c_start, c_end = windows.loc[campaign, ["START_DAY", "END_DAY"]]
        duration_weeks = windows.loc[campaign, "DURATION_DAYS"] / 7.0

        treated_hh = set(episodes.loc[episodes["CAMPAIGN"] == campaign, "HOUSEHOLD_KEY"]) & concurrent_free_hh

        def _overlaps_campaign(other_campaign: int) -> bool:
            o_start, o_end = all_campaign_windows.loc[other_campaign]
            return c_start <= o_end and o_start <= c_end

        overlapping_campaigns = {c for c in all_campaign_windows.index if c != campaign and _overlaps_campaign(c)}

        control_hh = {
            hh for hh, camps in linked_campaigns_by_hh.items()
            if hh not in treated_hh and not (camps & ({campaign} | overlapping_campaigns))
        }
        # Households present in the transaction data but never linked to any campaign are valid controls too.
        control_hh |= set(universal_outcomes["HOUSEHOLD_KEY"].unique()) - set(linked_campaigns_by_hh.keys()) - treated_hh

        treated_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(treated_hh))].copy()
        control_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(control_hh))].copy()

        if len(treated_rows) < 10 or len(control_rows) < 10:
            log.info(f"Campaign {campaign}: too few treated/control households after exclusions, skipping direct estimate (needs pooling)")
            results.append({"CAMPAIGN": campaign, "N_TREATED": len(treated_rows), "N_CONTROL": len(control_rows), "DID_ESTIMATE_PER_WEEK": np.nan, "NOTE": "insufficient sample -- requires hierarchical pooling"})
            continue

        combo = pd.concat([treated_rows.assign(_T=1), control_rows.assign(_T=0)], ignore_index=True)
        feature_cols = demo_cols + ["Y_PRE_SALES"]
        X = pd.get_dummies(combo[feature_cols], dummy_na=True).fillna(0)

        try:
            from sklearn.preprocessing import StandardScaler
            X_scaled = StandardScaler().fit_transform(X)
            ps_model = LogisticRegression(max_iter=2000)
            ps_model.fit(X_scaled, combo["_T"])
            combo["_pscore"] = ps_model.predict_proba(X_scaled)[:, 1]
        except Exception as e:
            log.warning(f"Campaign {campaign}: propensity model failed ({e}), using pre-period sales only for matching")
            combo["_pscore"] = combo["Y_PRE_SALES"]

        treated_idx = combo[combo["_T"] == 1].index
        control_idx = combo[combo["_T"] == 0].index
        nn = NearestNeighbors(n_neighbors=1).fit(combo.loc[control_idx, ["_pscore"]])
        _, match_pos = nn.kneighbors(combo.loc[treated_idx, ["_pscore"]])
        matched_control_idx = combo.loc[control_idx].iloc[match_pos.flatten()].index

        treated_during = combo.loc[treated_idx, outcome_col].to_numpy()
        treated_pre = combo.loc[treated_idx, "Y_PRE_SALES"].to_numpy()
        control_during = combo.loc[matched_control_idx, outcome_col].to_numpy()
        control_pre = combo.loc[matched_control_idx, "Y_PRE_SALES"].to_numpy()

        did_per_pair = (treated_during - treated_pre) - (control_during - control_pre)
        did_estimate = np.nanmean(did_per_pair) / max(duration_weeks, 1e-9)
        se = np.nanstd(did_per_pair, ddof=1) / np.sqrt(len(did_per_pair)) / max(duration_weeks, 1e-9)
        post_treated = combo.loc[treated_idx, "Y_POST_SALES"].to_numpy()
        post_control = combo.loc[matched_control_idx, "Y_POST_SALES"].to_numpy()
        post_did_per_pair = (post_treated - treated_pre) - (post_control - control_pre)
        post_did_estimate = np.nanmean(post_did_per_pair) / max(duration_weeks, 1e-9)
        results.append({
            "CAMPAIGN": campaign,
            "N_TREATED": len(treated_rows),
            "N_CONTROL": len(control_rows),
            "DID_ESTIMATE_PER_WEEK": did_estimate,
            "SE_PER_WEEK": se,
            "NOTE": "matched, single campaign -- pool across campaigns before reporting",
        })

    return pd.DataFrame(results)


# --------------------------------------------------------------------------- #
# 6. Hierarchical shrinkage and ranking (PDF Step 9)
# --------------------------------------------------------------------------- #

def pool_campaign_effects(plan1_results: pd.DataFrame, episodes: pd.DataFrame) -> pd.DataFrame:
    """
    Random-effects (DerSimonian-Laird) partial pooling of the per-campaign DiD
    estimates, nested by campaign type (TypeA/B/C from campaign_desc's
    DESCRIPTION field), addressing two problems the raw Plan 1 output has:

      1. Campaigns with too few treated/control households (DID_ESTIMATE_PER_WEEK
         is NaN) get no estimate at all in the raw output. Here they're assigned
         the pooled mean of same-type campaigns that DID get a direct estimate,
         falling back to the grand mean if their type has none either.
      2. Campaigns WITH a direct estimate are shrunk toward their type's pooled
         mean in proportion to how noisy they are (small SE -> little shrinkage,
         large SE -> shrunk hard toward the group). This is what the PDF calls
         "shrink noisy estimates toward the overall mean" and is a first step
         toward correcting the winner's curse -- it is NOT the full
         selection-corrected expectation the PDF asks for at final reporting
         (that needs a proper post-selection inference step, not implemented
         here), but it stops small campaigns from ranking artificially high.

    Returns plan1_results with added columns: CAMPAIGN_TYPE, GROUP_MEAN,
    TAU2 (between-campaign variance within type), SHRUNK_ESTIMATE_PER_WEEK,
    IS_DIRECT_ESTIMATE (0 if this campaign had no usable sample and is fully
    pooled), and RANK (by SHRUNK_ESTIMATE_PER_WEEK, descending).
    """
    log.info("Pooling campaign effects (empirical-Bayes shrinkage by campaign type)")

    campaign_type = episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")["CAMPAIGN_TYPE"]
    df = plan1_results.copy()
    df["CAMPAIGN_TYPE"] = df["CAMPAIGN"].map(campaign_type)
    df["IS_DIRECT_ESTIMATE"] = df["DID_ESTIMATE_PER_WEEK"].notna().astype(int)

    def _pool_group(group: pd.DataFrame) -> pd.DataFrame:
        direct = group[group["IS_DIRECT_ESTIMATE"] == 1]
        if len(direct) == 0:
            group["GROUP_MEAN"] = np.nan
            group["TAU2"] = np.nan
            group["SHRUNK_ESTIMATE_PER_WEEK"] = np.nan
            return group

        y = direct["DID_ESTIMATE_PER_WEEK"].to_numpy()
        se = direct["SE_PER_WEEK"].replace(0, np.nan).to_numpy()
        se = np.where(np.isnan(se), np.nanmedian(se) if not np.all(np.isnan(se)) else 1.0, se)
        w = 1.0 / (se ** 2)

        y_bar = np.sum(w * y) / np.sum(w)
        if len(y) > 1:
            Q = np.sum(w * (y - y_bar) ** 2)
            C = np.sum(w) - np.sum(w ** 2) / np.sum(w)
            tau2 = max(0.0, (Q - (len(y) - 1)) / C) if C > 0 else 0.0
        else:
            tau2 = 0.0  # single-campaign type: no between-campaign variance to estimate

        group["GROUP_MEAN"] = y_bar
        group["TAU2"] = tau2

        def _shrink(row):
            if row["IS_DIRECT_ESTIMATE"] == 0:
                return y_bar  # no direct estimate: fully pooled to group mean
            se_i2 = row["SE_PER_WEEK"] ** 2 if pd.notna(row["SE_PER_WEEK"]) and row["SE_PER_WEEK"] > 0 else np.nanmedian(se) ** 2
            if tau2 == 0:
                return y_bar
            lam = tau2 / (tau2 + se_i2)  # weight on the campaign's own estimate
            return lam * row["DID_ESTIMATE_PER_WEEK"] + (1 - lam) * y_bar

        group["SHRUNK_ESTIMATE_PER_WEEK"] = group.apply(_shrink, axis=1)
        return group

    pooled = df.groupby("CAMPAIGN_TYPE", group_keys=False).apply(_pool_group)

    # Fallback: any campaign type with zero direct estimates in it gets the
    # grand mean across ALL campaigns with direct estimates, not left as NaN.
    grand_mean = df.loc[df["IS_DIRECT_ESTIMATE"] == 1, "DID_ESTIMATE_PER_WEEK"].mean()
    pooled["SHRUNK_ESTIMATE_PER_WEEK"] = pooled["SHRUNK_ESTIMATE_PER_WEEK"].fillna(grand_mean)
    pooled["GROUP_MEAN"] = pooled["GROUP_MEAN"].fillna(grand_mean)

    pooled = pooled.sort_values("SHRUNK_ESTIMATE_PER_WEEK", ascending=False).reset_index(drop=True)
    pooled["RANK"] = pooled.index + 1

    log.info(
        f"Pooling complete: {pooled['IS_DIRECT_ESTIMATE'].sum()} campaigns had a direct "
        f"estimate, {len(pooled) - pooled['IS_DIRECT_ESTIMATE'].sum()} were fully pooled "
        f"from their type group or the grand mean."
    )
    return pooled


# --------------------------------------------------------------------------- #
# 7. Placebo test (PDF Step 6 -- validate the counterfactual)
# --------------------------------------------------------------------------- #

def run_placebo_test(
    episodes: pd.DataFrame,
    universal_outcomes: pd.DataFrame,
    pooled_results: pd.DataFrame,
    outcome_col: str = "Y_ELIGIBLE_SALES",
    n_draws_per_campaign: int = 200,
    seed: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Assigns FAKE treatment within the pool of households that were NEVER linked
    to ANY real campaign, using each real campaign's actual window and sample
    size, and computes the same DiD statistic. Repeated many times per campaign.
    Since no real treatment exists in this pool, the resulting distribution is
    a null: what DID_ESTIMATE_PER_WEEK values arise from pure sampling noise
    given this data's structure.

    This does NOT re-run the propensity-score matching from Plan 1 (that would
    be n_draws_per_campaign x 30 x logistic-regression-fits -- too slow for
    what's needed here). It uses a simple mean-difference DiD on random splits
    instead, which is the right tool for characterizing a null distribution
    (matching mainly helps point estimates, not the noise floor).

    Returns:
      placebo_draws       : every individual placebo draw (for inspection/plotting)
      pooled_with_pvalues : pooled_results with an added EMPIRICAL_P_VALUE column,
                             the two-sided share of ALL placebo draws (pooled
                             across campaigns, for a stabler tail estimate) at
                             least as extreme as that campaign's shrunk estimate.
    """
    log.info(f"Running placebo test: {n_draws_per_campaign} draws per campaign on never-linked households")
    rng = np.random.default_rng(seed)

    linked_hh = set(episodes["HOUSEHOLD_KEY"].unique())
    never_linked_hh = np.array(sorted(set(universal_outcomes["HOUSEHOLD_KEY"].unique()) - linked_hh))
    log.info(f"{len(never_linked_hh)} households were never linked to any real campaign -- placebo pool")

    duration_weeks_by_campaign = (
        episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")["DURATION_DAYS"] / 7.0
    )

    draws = []
    for _, row in pooled_results.drop_duplicates("CAMPAIGN").iterrows():
        campaign = row["CAMPAIGN"]
        n_treated = int(row["N_TREATED"]) if pd.notna(row["N_TREATED"]) and row["N_TREATED"] > 0 else 30
        duration_weeks = duration_weeks_by_campaign.get(campaign, 4.0)

        pool = universal_outcomes[
            (universal_outcomes["CAMPAIGN"] == campaign)
            & (universal_outcomes["HOUSEHOLD_KEY"].isin(never_linked_hh))
        ]
        if len(pool) < 20:
            continue  # not enough never-linked households with data for this campaign to draw from

        n_treated_draw = min(n_treated, len(pool) // 2)

        for _ in range(n_draws_per_campaign):
            shuffled = rng.permutation(pool["HOUSEHOLD_KEY"].to_numpy())
            fake_treated_hh = set(shuffled[:n_treated_draw])
            fake_control_hh = set(shuffled[n_treated_draw:])

            ft = pool[pool["HOUSEHOLD_KEY"].isin(fake_treated_hh)]
            fc = pool[pool["HOUSEHOLD_KEY"].isin(fake_control_hh)]

            treated_delta = ft[outcome_col].mean() - ft["Y_PRE_SALES"].mean()
            control_delta = fc[outcome_col].mean() - fc["Y_PRE_SALES"].mean()
            placebo_did = (treated_delta - control_delta) / max(duration_weeks, 1e-9)

            draws.append({"CAMPAIGN": campaign, "PLACEBO_DID_PER_WEEK": placebo_did})

    placebo_draws = pd.DataFrame(draws)
    if placebo_draws.empty:
        log.warning("Placebo test produced no draws -- likely too few never-linked households in the transaction file.")
        pooled_results["EMPIRICAL_P_VALUE"] = np.nan
        return placebo_draws, pooled_results

    null_values = placebo_draws["PLACEBO_DID_PER_WEEK"].to_numpy()
    log.info(
        f"Placebo null distribution: n={len(null_values)}, mean={null_values.mean():.4f}, "
        f"sd={null_values.std():.4f} (mean should be near 0 if the method is unbiased)"
    )

    def _p_value(estimate):
        if pd.isna(estimate):
            return np.nan
        return float(np.mean(np.abs(null_values) >= np.abs(estimate)))

    pooled_with_pvalues = pooled_results.copy()
    pooled_with_pvalues["EMPIRICAL_P_VALUE"] = pooled_with_pvalues["SHRUNK_ESTIMATE_PER_WEEK"].apply(_p_value)

    return placebo_draws, pooled_with_pvalues


# --------------------------------------------------------------------------- #
# 6. Orchestration
# --------------------------------------------------------------------------- #

def run_pipeline(cfg: Config) -> dict:
    tables = load_reference_tables(cfg)
    episodes = build_episode_table(tables)
    audit_summary = audit_transactions(cfg)

    # Determine which households we need window-outcomes for: everyone who
    # appears in the transaction file, so control candidates are available too.
    # Cheap pass just to collect the household id universe before the full compute.
    households_in_scope = set()
    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize, usecols=["household_key"]):
        households_in_scope |= set(chunk["household_key"].unique())

    universal_outcomes = compute_household_campaign_outcomes(
        cfg,
        tables["campaign_desc"],
        dedupe_coupon_table(tables["coupon"]),
        tables["product"],
        households_in_scope,
    )

    # Attach the treated-side outcomes onto the episode table for reference/export.
    episodes_with_outcomes = episodes.merge(
        universal_outcomes, on=["HOUSEHOLD_KEY", "CAMPAIGN"], how="left"
    )
    outcome_cols = [c for c in universal_outcomes.columns if c.startswith("Y_")]
    episodes_with_outcomes[outcome_cols] = episodes_with_outcomes[outcome_cols].fillna(0.0)

    plan1_results = run_plan1_event_study_did(episodes, universal_outcomes, tables["hh_demographic"])
    pooled_results = pool_campaign_effects(plan1_results, episodes)
    placebo_draws, pooled_results = run_placebo_test(episodes, universal_outcomes, pooled_results)

    return {
        "episodes": episodes_with_outcomes,
        "audit_summary": audit_summary,
        "plan1_results": plan1_results,
        "pooled_results": pooled_results,
        "placebo_draws": placebo_draws,
    }


def main(data_dir: str, out_dir: str = "./pipeline_output"):
    """Callable directly from a notebook: main('/path/to/csvs', './pipeline_output')"""
    cfg = Config(data_dir=Path(data_dir))
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    outputs = run_pipeline(cfg)
    outputs["episodes"].to_csv(out_path / "episode_table_with_outcomes.csv", index=False)
    outputs["plan1_results"].to_csv(out_path / "plan1_did_results.csv", index=False)
    outputs["pooled_results"].to_csv(out_path / "pooled_campaign_effects.csv", index=False)
    outputs["placebo_draws"].to_csv(out_path / "placebo_null_distribution.csv", index=False)
    log.info(f"Done. Outputs written to {out_path}")
    return outputs


if __name__ == "__main__":
    import sys
    import argparse

    # Jupyter/IPython injects its own kernel-connection args into sys.argv,
    # which argparse chokes on. Detect that case and fall back to editable
    # variables below instead of forcing you to run this as a .py script.
    running_in_notebook = "ipykernel_launcher" in sys.argv[0] or "ipykernel" in sys.modules

    if running_in_notebook:
        log.info("Detected notebook environment -- skipping argparse. Edit DATA_DIR/OUT_DIR below and re-run this cell.")
        DATA_DIR = "data"
        OUT_DIR = "./pipeline_output"
        outputs = main(DATA_DIR, OUT_DIR)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--data-dir", type=str, required=True, help="Directory containing all 7 CSVs")
        parser.add_argument("--out-dir", type=str, default="./pipeline_output")
        args = parser.parse_args()
        outputs = main(args.data_dir, args.out_dir)

2026-08-06 12:51:57,348 | INFO | Detected notebook environment -- skipping argparse. Edit DATA_DIR/OUT_DIR below and re-run this cell.
2026-08-06 12:51:57,349 | INFO | Loading reference tables
2026-08-06 12:51:57,440 | INFO | Building episode table
2026-08-06 12:51:57,447 | INFO | coupon.csv: dropped 5164 exact-duplicate rows (4.15%)
2026-08-06 12:51:58,287 | INFO | Episode table: 7208 rows, 1584 households
2026-08-06 12:51:58,287 | INFO | Auditing transaction file (chunked)
2026-08-06 12:51:59,567 | INFO | Audit summary: {'total_rows': 2595732, 'total_quantity': 260685622.0, 'negative_sales_rows': 0, 'negative_quantity_rows': 0, 'top_1pct_qty_row_share': 0.010642470023869952, 'top_1pct_qty_unit_share': 0.9873510054958076, 'n_households': 2500, 'n_weeks': 102, 'household_week_no_trip_share': 0.5138196078431372}
2026-08-06 12:51:59,567 | WARNING | top_1pct_qty_unit_share above should be compared against the PDF's claimed 98.7%-units-in-1.2%-of-rows fuel-mixing figure. If far lower, this

In [32]:
# --------------------------------------------------------------------------- #
# Step 4.5: Repurchase-cycle estimation from real transaction data
# --------------------------------------------------------------------------- #
#
# Goal: derive an evidence-based post_period_weeks per commodity, instead of
# the fixed 4-week placeholder in Config. Per document 3's rule, the horizon
# must be at least as long as the product's natural purchase frequency, and
# per the PDF's baseline rule, this must be estimated ONLY on ordinary,
# unpromoted periods -- so we exclude any day that falls inside ANY campaign's
# window (not just the household's own linked campaigns), since a household
# can be exposed to promotional pricing/marketing spillover even from
# campaigns it isn't formally linked to.
def estimate_repurchase_cycles(cfg, campaign_desc, product, min_purchases=3):
    """
    Streams transaction_data.csv once. For each household x commodity pair,
    computes inter-purchase day gaps using ONLY unpromoted days -- i.e. days
    outside every campaign's [START_DAY, END_DAY] window.

    IMPORTANT FIX: earlier version restricted the whole search to the
    224-719 campaign-season window and THEN excluded promoted days within
    it -- since campaigns overlap heavily, that window can be almost fully
    covered by promotions, leaving zero unpromoted days. This version uses
    the FULL transaction history: days outside [day_min, day_max] are
    unpromoted by definition and need no lookup; only days inside that
    range go through the per-day promoted check.
    """
    log.info("Estimating repurchase cycles from unpromoted transaction days")

    windows = campaign_desc[["START_DAY", "END_DAY"]].to_numpy()
    day_min, day_max = int(campaign_desc["START_DAY"].min()), int(campaign_desc["END_DAY"].max())

    def _is_promoted_day(day):
        return np.any((windows[:, 0] <= day) & (day <= windows[:, 1]))

    all_days = np.arange(day_min, day_max + 1)
    promoted_lookup = pd.Series(
        [_is_promoted_day(d) for d in all_days], index=all_days
    )

    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]
    purchase_days = {}
    total_rows_seen = 0
    total_unpromoted_rows = 0

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        total_rows_seen += len(chunk)
        chunk["COMMODITY_DESC"] = chunk["PRODUCT_ID"].map(product_commodity)
        chunk = chunk.dropna(subset=["COMMODITY_DESC"])

        # Days inside the campaign season: check the lookup.
        # Days outside it: unpromoted by definition, keep automatically.
        inside_season = chunk["DAY"].between(day_min, day_max)
        is_promoted = pd.Series(False, index=chunk.index)
        if inside_season.any():
            is_promoted.loc[inside_season] = promoted_lookup.reindex(
                chunk.loc[inside_season, "DAY"]
            ).to_numpy()

        chunk = chunk[~is_promoted]
        total_unpromoted_rows += len(chunk)
        if chunk.empty:
            continue

        grp = chunk.groupby(["household_key", "COMMODITY_DESC"])["DAY"].apply(
            lambda s: sorted(s.unique())
        )
        for (hh, commodity), days in grp.items():
            key = (hh, commodity)
            purchase_days.setdefault(key, [])
            purchase_days[key].extend(days)

    log.info(f"Unpromoted rows kept: {total_unpromoted_rows} / {total_rows_seen} "
              f"({total_unpromoted_rows/total_rows_seen:.1%})")

    rows = []
    for (hh, commodity), days in purchase_days.items():
        days = sorted(set(days))
        if len(days) < min_purchases:
            continue
        gaps = np.diff(days)
        rows.append({"COMMODITY_DESC": commodity, "HOUSEHOLD_KEY": hh, "MEDIAN_GAP_DAYS": np.median(gaps)})

    gap_df = pd.DataFrame(rows)
    if gap_df.empty:
        log.warning("No household-commodity pairs met min_purchases threshold on unpromoted "
                     "days -- returning empty summary. Check total_unpromoted_rows above.")
        return pd.DataFrame(columns=["COMMODITY_DESC", "n_households", "median_gap", "p75_gap", "p90_gap"])

    summary = (
        gap_df.groupby("COMMODITY_DESC")["MEDIAN_GAP_DAYS"]
        .agg(n_households="count", median_gap="median",
             p75_gap=lambda s: np.percentile(s, 75),
             p90_gap=lambda s: np.percentile(s, 90))
        .reset_index()
        .sort_values("n_households", ascending=False)
    )
    log.info(f"Repurchase-cycle estimates computed for {len(summary)} commodities "
             f"(households with >= {min_purchases} unpromoted purchases only)")
    return summary
cfg = Config(Path("data"))
tables = load_reference_tables(cfg)
repurchase_cycles = estimate_repurchase_cycles(
    cfg, tables["campaign_desc"], tables["product"], min_purchases=3
)
repurchase_cycles.to_csv("pipeline_output/repurchase_cycles_by_commodity.csv", index=False)
repurchase_cycles.head(20)

2026-08-06 15:37:27,996 | INFO | Loading reference tables
2026-08-06 15:37:28,076 | INFO | Estimating repurchase cycles from unpromoted transaction days
2026-08-06 15:37:31,488 | INFO | Unpromoted rows kept: 627727 / 2595732 (24.2%)
2026-08-06 15:37:31,994 | INFO | Repurchase-cycle estimates computed for 270 commodities (households with >= 3 unpromoted purchases only)


,COMMODITY_DESC,n_households,median_gap,p75_gap,p90_gap
112,FLUID MILK PRODUCTS,1565,13.5,24.00,42.0
14,BAKED BREAD/BUNS/ROLLS,1502,14.0,24.00,39.5
239,SOFT DRINKS,1488,12.0,22.50,38.0
48,CHEESE,1284,16.5,29.00,46.5
13,BAG SNACKS,1181,16.5,28.00,46.0
23,BEEF,1082,16.0,28.00,46.5
93,EGGS,855,22.0,36.00,53.0
256,TROPICAL FRUIT,853,16.0,28.00,45.0
61,COLD CEREAL,799,20.0,32.00,50.0
213,REFRGRATD JUICES/DRNKS,777,17.0,30.00,50.5


In [33]:
# --------------------------------------------------------------------------- #
# Step 4.6: Translate per-commodity repurchase cycles into a per-campaign
# post-window, then re-run outcome construction and Plan 1 with it.
# --------------------------------------------------------------------------- #
#
# Logic: each campaign's eligible products belong to a set of commodities
# (already computed inside compute_household_campaign_outcomes as
# eligible_commodities_by_campaign). The post-window for that campaign should
# be long enough to catch pull-forward for the SLOWEST-cycling eligible
# commodity, not the fastest -- otherwise pantry-loading on long-cycle goods
# gets cut off and Y_POST_SALES understates the payback effect for exactly
# the products most prone to it. We use the p75 gap (not the median) as the
# per-commodity horizon, per document 3's rule that the horizon must be AT
# LEAST as long as natural purchase frequency -- median would leave half of
# repurchases outside the window by construction.

def build_campaign_post_windows(repurchase_cycles, coupon, product, default_weeks=4, cap_weeks=12):
    """
    Returns {CAMPAIGN: post_period_weeks}, one value per campaign, derived as
    the max p75_gap (in weeks, rounded up) across that campaign's eligible
    commodities. Falls back to default_weeks if a campaign's commodities have
    no repurchase-cycle estimate (too few unpromoted-day observations).
    Capped at cap_weeks to prevent one long-cycle outlier commodity (e.g.
    a durable good bought twice a year) from blowing up the post-window for
    an entire campaign of otherwise fast-cycling products.
    """
    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]
    elig = coupon[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    elig["COMMODITY_DESC"] = elig["PRODUCT_ID"].map(product_commodity)

    gap_lookup = repurchase_cycles.set_index("COMMODITY_DESC")["p75_gap"]

    campaign_windows = {}
    campaigns_using_default = []
    for campaign, group in elig.groupby("CAMPAIGN"):
        commodities = set(group["COMMODITY_DESC"].dropna())
        gaps = gap_lookup.reindex(list(commodities)).dropna()
        if gaps.empty:
            campaign_windows[campaign] = default_weeks
            campaigns_using_default.append(campaign)
        else:
            weeks = int(np.ceil(gaps.max() / 7.0))
            campaign_windows[campaign] = min(weeks, cap_weeks)

    if campaigns_using_default:
        log.warning(f"{len(campaigns_using_default)} campaigns had no commodity-level repurchase "
                     f"estimate and fell back to the {default_weeks}-week default: {campaigns_using_default}")

    log.info(f"Per-campaign post-windows (weeks): min={min(campaign_windows.values())}, "
             f"median={np.median(list(campaign_windows.values())):.1f}, "
             f"max={max(campaign_windows.values())}")
    return campaign_windows


campaign_post_windows = build_campaign_post_windows(
    repurchase_cycles, dedupe_coupon_table(tables["coupon"]), tables["product"]
)
pd.Series(campaign_post_windows, name="post_period_weeks").rename_axis("CAMPAIGN").to_csv(
    "pipeline_output/campaign_post_windows.csv"
)
campaign_post_windows

2026-08-06 15:45:28,759 | INFO | coupon.csv: dropped 5164 exact-duplicate rows (4.15%)
2026-08-06 15:45:28,801 | INFO | Per-campaign post-windows (weeks): min=5, median=8.0, max=10


{1: 6,
 2: 7,
 3: 8,
 4: 7,
 5: 10,
 6: 5,
 7: 8,
 8: 10,
 9: 8,
 10: 8,
 11: 7,
 12: 7,
 13: 10,
 14: 8,
 15: 5,
 16: 8,
 17: 8,
 18: 10,
 19: 7,
 20: 8,
 21: 8,
 22: 9,
 23: 8,
 24: 7,
 25: 7,
 26: 10,
 27: 8,
 28: 7,
 29: 8,
 30: 8}